In [2]:
import arviz as az
import pymc as pm
from matplotlib import pyplot as plt
import hssm
import pandas as pd
from hssm.likelihoods import DDM
import pytensor 
import pytensor.tensor as pt
from pytensor.graph.op import Op 
import pickle as pkl 
import numpy as np
import time 
from scipy.optimize import fmin, minimize, LinearConstraint, Bounds 
from efficient_fpt.multi_stage_cy import (compute_loss_parallel, print_num_threads, compute_tadaloss_parallel)
import jax 
import jax.numpy as jnp 
from jax import grad,jit,vmap 

print(f"PyMC version: {pm.__version__}")
print(f"JAX version: {jax.__version__}")
print(f"JAX devices: {jax.devices()}")

PyMC version: 5.28.4
JAX version: 0.9.2
JAX devices: [CudaDevice(id=0), CudaDevice(id=1), CudaDevice(id=2), CudaDevice(id=3)]


In [5]:
from hssm.distribution_utils import make_distribution_for_supported_model

ANGLE = make_distribution_for_supported_model("angle", loglik_kind="approx_differentiable")

v_true, a_true, z_true, t_true, sv_true = [0.5, 1.5, 0.5, 0.5, 0.1]
dataset = hssm.simulate_data(
    model="ddm_sdv",
    theta=[v_true, a_true, z_true, t_true, sv_true],
    size=1000,
)


with pm.Model() as angle_pymc:
    v = pm.Uniform("v", lower=-10.0, upper=10.0)
    a = pm.HalfNormal("a", sigma=2.0)
    z = pm.Uniform("z", lower=0.01, upper=0.99)
    t = pm.Uniform("t", lower=0.0, upper=0.6, initval=0.1)
    theta = pm.Uniform("theta", lower=0, upper=0.5)

    angle = ANGLE(
        "angle",
        v=v,
        a=a,
        z=z,
        t=t,
        theta=theta,
        observed=dataset.values,
    )

    # angle_pymc_trace = pm.sample(mp_ctx="spawn", tune=200, draws=200)
